In [ ]:
import pandas as pd
import requests
from pathlib import Path
import duckdb

# Data Profiling on Bronze Data

Prerequisites:

data_loader -> bronze_chicago_taxi_2024.parquet
            -> bronze_weatherdata.parquet

## Taxi Data

In [ ]:
TAXI_PATH = "bronze_chicago_taxi_2024.parquet"

duckdb.sql(f"""
SELECT COUNT(*) AS n_rows
FROM read_parquet('{TAXI_PATH}')
""").show()

duckdb.sql(f"""
DESCRIBE SELECT *
FROM read_parquet('{TAXI_PATH}')
""").show(max_rows=100)

┌──────────┐
│  n_rows  │
│  int64   │
├──────────┤
│ 15406960 │
└──────────┘

┌────────────────────────────┬──────────────────────────┬─────────┬─────────┬─────────┬─────────┐
│        column_name         │       column_type        │  null   │   key   │ default │  extra  │
│          varchar           │         varchar          │ varchar │ varchar │ varchar │ varchar │
├────────────────────────────┼──────────────────────────┼─────────┼─────────┼─────────┼─────────┤
│ :id                        │ VARCHAR                  │ YES     │ NULL    │ NULL    │ NULL    │
│ :version                   │ VARCHAR                  │ YES     │ NULL    │ NULL    │ NULL    │
│ :created_at                │ TIMESTAMP WITH TIME ZONE │ YES     │ NULL    │ NULL    │ NULL    │
│ :updated_at                │ TIMESTAMP WITH TIME ZONE │ YES     │ NULL    │ NULL    │ NULL    │
│ trip_id                    │ VARCHAR                  │ YES     │ NULL    │ NULL    │ NULL    │
│ taxi_id                    │ VARCHAR 

## Null Values

In [ ]:
con = duckdb.connect()

columns = con.sql(f"""
DESCRIBE SELECT *
FROM read_parquet('{TAXI_PATH}')
""").df()["column_name"].tolist()

parts = []

for col in columns:
    parts.append(f"""
    SELECT
        '{col}' AS column_name,
        COUNT(*) AS n_rows,
        SUM(CASE WHEN "{col}" IS NULL THEN 1 ELSE 0 END) AS n_nulls,
        ROUND(
            100.0 * SUM(CASE WHEN "{col}" IS NULL THEN 1 ELSE 0 END) / COUNT(*),
            2
        ) AS null_pct
    FROM read_parquet('{TAXI_PATH}')
    """)

query = "\nUNION ALL\n".join(parts) + "\nORDER BY null_pct DESC"

null_report = con.sql(query).df()
print(null_report.to_string(index=False))

               column_name   n_rows   n_nulls  null_pct
      dropoff_census_tract 15406960 8751654.0     56.80
       pickup_census_tract 15406960 8547337.0     55.48
    dropoff_community_area 15406960 1366538.0      8.87
 dropoff_centroid_location 15406960 1285413.0      8.34
 dropoff_centroid_latitude 15406960 1285413.0      8.34
dropoff_centroid_longitude 15406960 1285413.0      8.34
     pickup_community_area 15406960  429216.0      2.79
  pickup_centroid_latitude 15406960  421354.0      2.73
 pickup_centroid_longitude 15406960  421354.0      2.73
  pickup_centroid_location 15406960  421354.0      2.73
                      fare 15406960   31816.0      0.21
                      tips 15406960   31816.0      0.21
                     tolls 15406960   31816.0      0.21
                    extras 15406960   31816.0      0.21
                trip_total 15406960   31816.0      0.21
              trip_seconds 15406960    2936.0      0.02
                   trip_id 15406960       0.0   

In [ ]:
duckdb.sql(f"""
    SELECT 
        COUNT(*)
    FROM read_parquet('{TAXI_PATH}')
    WHERE
        pickup_centroid_location IS NULL
        AND pickup_centroid_longitude IS NULL
        AND pickup_centroid_latitude IS NULL
        AND pickup_census_tract IS NULL
""")

    

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       419331 │
└──────────────┘

## Plausibility Checks

Columns

Trip ID
Taxi ID 
Trip Start Timestamp - check if equal or earlier then "trip end timestamp", if "trip seconds" > 15 min
Trip End Timestamp - check if equal or greater then "trip start timestamp", if "trip seconds" > 15 min
Trip Seconds - check if greater then 0 and not less then "trip end timestamp" - "trip start timestamp", look for outliers
Trip Miles - check if greater then 0, look for outliers
Pickup Census Tract -> can be dropped and reestimated using latitude and longitude
Dropoff Census Tract -> can be dropped and reestimated using latitude and longitude
Pickup Community Area
Dropoff Community Area
Fare - check if equal or greater then 0, look for outliers
Tips - check if greater then 0, look for outliers
Tolls - check if greater then 0, look for outliers
Extras - check if greater then 0, look for outliers
Trip Total - check if greater then 0, look for outliers
Payment Type
Company
Pickup Centroid Latitude - check if coordinate is reasonable for Chicago (in BROAD range of chicago)
Pickup Centroid Longitude - check if coordinate is reasonable for Chicago (in BROAD range of chicago)
Pickup Centroid Location
Dropoff Centroid Latitude - check if coordinate is reasonable for Chicago (in BROAD range of chicago)
Dropoff Centroid Longitude - check if coordinate is reasonable for Chicago (in BROAD range of chicago)
Dropoff Centroid Location


In [ ]:
plausi_report = duckdb.sql(f"""
WITH base AS (
    SELECT
        *,
        TRY_CAST(trip_start_timestamp AS TIMESTAMP) AS trip_start,
        TRY_CAST(trip_end_timestamp AS TIMESTAMP) AS trip_end,

        DATE_DIFF(
            'second',
            TRY_CAST(trip_start_timestamp AS TIMESTAMP),
            TRY_CAST(trip_end_timestamp AS TIMESTAMP)
        ) AS timestamp_diff_seconds,

        trip_seconds / 60.0 AS trip_minutes,

        trip_miles / NULLIF(trip_seconds / 3600.0, 0) AS mph,

        fare / NULLIF(trip_miles, 0) AS fare_per_mile,
        tips / NULLIF(fare, 0) AS tip_rate,
        trip_total / NULLIF(trip_miles, 0) AS total_per_mile,

        ABS(
            trip_total
            - (
                COALESCE(fare, 0)
                + COALESCE(tips, 0)
                + COALESCE(tolls, 0)
                + COALESCE(extras, 0)
            )
        ) AS trip_total_component_diff

    FROM read_parquet('{TAXI_PATH}')
),

checks AS (
    SELECT
        COUNT(*) AS total_rows,

        -- IDs
        SUM(CASE WHEN trip_id IS NULL THEN 1 ELSE 0 END) AS trip_id_null,
        COUNT(*) - COUNT(DISTINCT trip_id) AS trip_id_duplicate,

        SUM(CASE WHEN taxi_id IS NULL THEN 1 ELSE 0 END) AS taxi_id_null,

        -- timestamps
        SUM(CASE WHEN trip_start IS NULL THEN 1 ELSE 0 END) AS trip_start_parse_failed,
        SUM(CASE WHEN trip_end IS NULL THEN 1 ELSE 0 END) AS trip_end_parse_failed,

        SUM(CASE
            WHEN trip_start IS NOT NULL
             AND trip_end IS NOT NULL
             AND trip_start > trip_end
            THEN 1 ELSE 0
        END) AS trip_start_after_trip_end,

        SUM(CASE
            WHEN trip_start IS NOT NULL
             AND trip_end IS NOT NULL
             AND trip_seconds > 900
             AND trip_start >= trip_end
            THEN 1 ELSE 0
        END) AS trip_start_not_before_end_for_trips_gt_15min,

        SUM(CASE
            WHEN trip_start IS NOT NULL
             AND trip_end IS NOT NULL
             AND trip_seconds > 900
             AND trip_end <= trip_start
            THEN 1 ELSE 0
        END) AS trip_end_not_after_start_for_trips_gt_15min,

        -- trip_seconds
        SUM(CASE WHEN trip_seconds IS NULL THEN 1 ELSE 0 END) AS trip_seconds_null,
        SUM(CASE WHEN trip_seconds <= 0 THEN 1 ELSE 0 END) AS trip_seconds_lte_0,

        SUM(CASE
            WHEN trip_seconds IS NOT NULL
             AND timestamp_diff_seconds IS NOT NULL
             AND trip_seconds < timestamp_diff_seconds
            THEN 1 ELSE 0
        END) AS trip_seconds_lt_timestamp_diff,

        SUM(CASE
            WHEN trip_seconds IS NOT NULL
             AND timestamp_diff_seconds IS NOT NULL
             AND ABS(trip_seconds - timestamp_diff_seconds) > 900
            THEN 1 ELSE 0
        END) AS trip_seconds_timestamp_diff_abs_gt_15min,

        SUM(CASE WHEN trip_seconds > 10800 THEN 1 ELSE 0 END) AS trip_seconds_gt_3h,
        SUM(CASE WHEN trip_seconds > 21600 THEN 1 ELSE 0 END) AS trip_seconds_gt_6h,

        -- trip_miles
        SUM(CASE WHEN trip_miles IS NULL THEN 1 ELSE 0 END) AS trip_miles_null,
        SUM(CASE WHEN trip_miles <= 0 THEN 1 ELSE 0 END) AS trip_miles_lte_0,
        SUM(CASE WHEN trip_miles > 100 THEN 1 ELSE 0 END) AS trip_miles_gt_100,
        SUM(CASE WHEN trip_miles > 250 THEN 1 ELSE 0 END) AS trip_miles_gt_250,

        -- census tracts
        SUM(CASE WHEN pickup_census_tract IS NULL THEN 1 ELSE 0 END) AS pickup_census_tract_null,
        SUM(CASE WHEN dropoff_census_tract IS NULL THEN 1 ELSE 0 END) AS dropoff_census_tract_null,

        -- community areas
        SUM(CASE WHEN pickup_community_area IS NULL THEN 1 ELSE 0 END) AS pickup_community_area_null,
        SUM(CASE
            WHEN pickup_community_area IS NOT NULL
             AND (pickup_community_area < 1 OR pickup_community_area > 77)
            THEN 1 ELSE 0
        END) AS pickup_community_area_outside_1_77,

        SUM(CASE WHEN dropoff_community_area IS NULL THEN 1 ELSE 0 END) AS dropoff_community_area_null,
        SUM(CASE
            WHEN dropoff_community_area IS NOT NULL
             AND (dropoff_community_area < 1 OR dropoff_community_area > 77)
            THEN 1 ELSE 0
        END) AS dropoff_community_area_outside_1_77,

        -- fare
        SUM(CASE WHEN fare IS NULL THEN 1 ELSE 0 END) AS fare_null,
        SUM(CASE WHEN fare <= 0 THEN 1 ELSE 0 END) AS fare_lte_0,
        SUM(CASE WHEN fare > 500 THEN 1 ELSE 0 END) AS fare_gt_500,
        SUM(CASE WHEN fare > 1000 THEN 1 ELSE 0 END) AS fare_gt_1000,

        -- tips
        SUM(CASE WHEN tips IS NULL THEN 1 ELSE 0 END) AS tips_null,
        SUM(CASE WHEN tips < 0 THEN 1 ELSE 0 END) AS tips_negative,
        SUM(CASE WHEN tips > 200 THEN 1 ELSE 0 END) AS tips_gt_200,
        SUM(CASE WHEN tip_rate > 1 THEN 1 ELSE 0 END) AS tip_rate_gt_100pct,

        -- tolls
        SUM(CASE WHEN tolls IS NULL THEN 1 ELSE 0 END) AS tolls_null,
        SUM(CASE WHEN tolls < 0 THEN 1 ELSE 0 END) AS tolls_negative,
        SUM(CASE WHEN tolls > 100 THEN 1 ELSE 0 END) AS tolls_gt_100,

        -- extras
        SUM(CASE WHEN extras IS NULL THEN 1 ELSE 0 END) AS extras_null,
        SUM(CASE WHEN extras < 0 THEN 1 ELSE 0 END) AS extras_negative,
        SUM(CASE WHEN extras > 100 THEN 1 ELSE 0 END) AS extras_gt_100,

        -- trip_total
        SUM(CASE WHEN trip_total IS NULL THEN 1 ELSE 0 END) AS trip_total_null,
        SUM(CASE WHEN trip_total <= 0 THEN 1 ELSE 0 END) AS trip_total_lte_0,
        SUM(CASE WHEN trip_total > 1000 THEN 1 ELSE 0 END) AS trip_total_gt_1000,
        SUM(CASE WHEN trip_total_component_diff > 0.01 THEN 1 ELSE 0 END) AS trip_total_component_mismatch_gt_1_cent,

        -- payment type
        SUM(CASE WHEN payment_type IS NULL THEN 1 ELSE 0 END) AS payment_type_null,
        SUM(CASE WHEN TRIM(payment_type) = '' THEN 1 ELSE 0 END) AS payment_type_empty,

        -- company
        SUM(CASE WHEN company IS NULL THEN 1 ELSE 0 END) AS company_null,
        SUM(CASE WHEN TRIM(company) = '' THEN 1 ELSE 0 END) AS company_empty,

        -- pickup latitude / longitude
        SUM(CASE WHEN pickup_centroid_latitude IS NULL THEN 1 ELSE 0 END) AS pickup_lat_null,
        SUM(CASE
            WHEN pickup_centroid_latitude IS NOT NULL
             AND (pickup_centroid_latitude < 41 OR pickup_centroid_latitude > 43)
            THEN 1 ELSE 0
        END) AS pickup_lat_outside_broad_chicago_range,

        SUM(CASE WHEN pickup_centroid_longitude IS NULL THEN 1 ELSE 0 END) AS pickup_lon_null,
        SUM(CASE
            WHEN pickup_centroid_longitude IS NOT NULL
             AND (pickup_centroid_longitude < -89 OR pickup_centroid_longitude > -87)
            THEN 1 ELSE 0
        END) AS pickup_lon_outside_broad_chicago_range,

        SUM(CASE WHEN pickup_centroid_location IS NULL THEN 1 ELSE 0 END) AS pickup_location_null,

        -- dropoff latitude / longitude
        SUM(CASE WHEN dropoff_centroid_latitude IS NULL THEN 1 ELSE 0 END) AS dropoff_lat_null,
        SUM(CASE
            WHEN dropoff_centroid_latitude IS NOT NULL
             AND (dropoff_centroid_latitude < 41 OR dropoff_centroid_latitude > 43)
            THEN 1 ELSE 0
        END) AS dropoff_lat_outside_broad_chicago_range,

        SUM(CASE WHEN dropoff_centroid_longitude IS NULL THEN 1 ELSE 0 END) AS dropoff_lon_null,
        SUM(CASE
            WHEN dropoff_centroid_longitude IS NOT NULL
             AND (dropoff_centroid_longitude < -89 OR dropoff_centroid_longitude > -87)
            THEN 1 ELSE 0
        END) AS dropoff_lon_outside_broad_chicago_range,

        SUM(CASE WHEN dropoff_centroid_location IS NULL THEN 1 ELSE 0 END) AS dropoff_location_null,

        -- derived plausibility
        SUM(CASE WHEN mph > 100 THEN 1 ELSE 0 END) AS mph_gt_100,
        SUM(CASE WHEN fare_per_mile > 50 THEN 1 ELSE 0 END) AS fare_per_mile_gt_50,
        SUM(CASE WHEN total_per_mile > 100 THEN 1 ELSE 0 END) AS total_per_mile_gt_100

    FROM base
),

long AS (
    -- Trip ID
    SELECT 'trip_id' AS column_name, 'is_null' AS check_name, trip_id_null AS n_rows, total_rows, 'high' AS severity FROM checks
    UNION ALL SELECT 'trip_id', 'duplicate', trip_id_duplicate, total_rows, 'high' FROM checks

    -- Taxi ID
    UNION ALL SELECT 'taxi_id', 'is_null', taxi_id_null, total_rows, 'medium' FROM checks

    -- Trip Start Timestamp
    UNION ALL SELECT 'trip_start_timestamp', 'parse_failed', trip_start_parse_failed, total_rows, 'high' FROM checks
    UNION ALL SELECT 'trip_start_timestamp', 'after_trip_end_timestamp', trip_start_after_trip_end, total_rows, 'high' FROM checks
    UNION ALL SELECT 'trip_start_timestamp', 'not_before_end_when_trip_seconds_gt_15min', trip_start_not_before_end_for_trips_gt_15min, total_rows, 'high' FROM checks

    -- Trip End Timestamp
    UNION ALL SELECT 'trip_end_timestamp', 'parse_failed', trip_end_parse_failed, total_rows, 'high' FROM checks
    UNION ALL SELECT 'trip_end_timestamp', 'not_after_start_when_trip_seconds_gt_15min', trip_end_not_after_start_for_trips_gt_15min, total_rows, 'high' FROM checks

    -- Trip Seconds
    UNION ALL SELECT 'trip_seconds', 'is_null', trip_seconds_null, total_rows, 'high' FROM checks
    UNION ALL SELECT 'trip_seconds', '<= 0', trip_seconds_lte_0, total_rows, 'high' FROM checks
    UNION ALL SELECT 'trip_seconds', '< timestamp_diff_seconds', trip_seconds_lt_timestamp_diff, total_rows, 'medium' FROM checks
    UNION ALL SELECT 'trip_seconds', 'abs(trip_seconds - timestamp_diff_seconds) > 15min', trip_seconds_timestamp_diff_abs_gt_15min, total_rows, 'medium' FROM checks
    UNION ALL SELECT 'trip_seconds', '> 3h', trip_seconds_gt_3h, total_rows, 'medium' FROM checks
    UNION ALL SELECT 'trip_seconds', '> 6h', trip_seconds_gt_6h, total_rows, 'high' FROM checks

    -- Trip Miles
    UNION ALL SELECT 'trip_miles', 'is_null', trip_miles_null, total_rows, 'high' FROM checks
    UNION ALL SELECT 'trip_miles', '<= 0', trip_miles_lte_0, total_rows, 'high' FROM checks
    UNION ALL SELECT 'trip_miles', '> 100 miles', trip_miles_gt_100, total_rows, 'medium' FROM checks
    UNION ALL SELECT 'trip_miles', '> 250 miles', trip_miles_gt_250, total_rows, 'high' FROM checks

    -- Census Tract
    UNION ALL SELECT 'pickup_census_tract', 'is_null', pickup_census_tract_null, total_rows, 'info' FROM checks
    UNION ALL SELECT 'dropoff_census_tract', 'is_null', dropoff_census_tract_null, total_rows, 'info' FROM checks

    -- Community Area
    UNION ALL SELECT 'pickup_community_area', 'is_null', pickup_community_area_null, total_rows, 'medium' FROM checks
    UNION ALL SELECT 'pickup_community_area', 'outside 1..77', pickup_community_area_outside_1_77, total_rows, 'high' FROM checks
    UNION ALL SELECT 'dropoff_community_area', 'is_null', dropoff_community_area_null, total_rows, 'medium' FROM checks
    UNION ALL SELECT 'dropoff_community_area', 'outside 1..77', dropoff_community_area_outside_1_77, total_rows, 'high' FROM checks

    -- Fare
    UNION ALL SELECT 'fare', 'is_null', fare_null, total_rows, 'high' FROM checks
    UNION ALL SELECT 'fare', '<= 0', fare_lte_0, total_rows, 'high' FROM checks
    UNION ALL SELECT 'fare', '> 500', fare_gt_500, total_rows, 'medium' FROM checks
    UNION ALL SELECT 'fare', '> 1000', fare_gt_1000, total_rows, 'high' FROM checks

    -- Tips
    UNION ALL SELECT 'tips', 'is_null', tips_null, total_rows, 'medium' FROM checks
    UNION ALL SELECT 'tips', '< 0', tips_negative, total_rows, 'high' FROM checks
    UNION ALL SELECT 'tips', '> 200', tips_gt_200, total_rows, 'medium' FROM checks
    UNION ALL SELECT 'tips', 'tip_rate > 100%', tip_rate_gt_100pct, total_rows, 'medium' FROM checks

    -- Tolls
    UNION ALL SELECT 'tolls', 'is_null', tolls_null, total_rows, 'medium' FROM checks
    UNION ALL SELECT 'tolls', '< 0', tolls_negative, total_rows, 'high' FROM checks
    UNION ALL SELECT 'tolls', '> 100', tolls_gt_100, total_rows, 'medium' FROM checks

    -- Extras
    UNION ALL SELECT 'extras', 'is_null', extras_null, total_rows, 'medium' FROM checks
    UNION ALL SELECT 'extras', '< 0', extras_negative, total_rows, 'high' FROM checks
    UNION ALL SELECT 'extras', '> 100', extras_gt_100, total_rows, 'medium' FROM checks

    -- Trip Total
    UNION ALL SELECT 'trip_total', 'is_null', trip_total_null, total_rows, 'high' FROM checks
    UNION ALL SELECT 'trip_total', '<= 0', trip_total_lte_0, total_rows, 'high' FROM checks
    UNION ALL SELECT 'trip_total', '> 1000', trip_total_gt_1000, total_rows, 'medium' FROM checks
    UNION ALL SELECT 'trip_total', 'fare + tips + tolls + extras mismatch > 0.01', trip_total_component_mismatch_gt_1_cent, total_rows, 'medium' FROM checks

    -- Payment Type
    UNION ALL SELECT 'payment_type', 'is_null', payment_type_null, total_rows, 'medium' FROM checks
    UNION ALL SELECT 'payment_type', 'empty string', payment_type_empty, total_rows, 'medium' FROM checks

    -- Company
    UNION ALL SELECT 'company', 'is_null', company_null, total_rows, 'medium' FROM checks
    UNION ALL SELECT 'company', 'empty string', company_empty, total_rows, 'medium' FROM checks

    -- Pickup Coordinates
    UNION ALL SELECT 'pickup_centroid_latitude', 'is_null', pickup_lat_null, total_rows, 'medium' FROM checks
    UNION ALL SELECT 'pickup_centroid_latitude', 'outside broad Chicago range 41..43', pickup_lat_outside_broad_chicago_range, total_rows, 'high' FROM checks
    UNION ALL SELECT 'pickup_centroid_longitude', 'is_null', pickup_lon_null, total_rows, 'medium' FROM checks
    UNION ALL SELECT 'pickup_centroid_longitude', 'outside broad Chicago range -89..-87', pickup_lon_outside_broad_chicago_range, total_rows, 'high' FROM checks
    UNION ALL SELECT 'pickup_centroid_location', 'is_null', pickup_location_null, total_rows, 'medium' FROM checks

    -- Dropoff Coordinates
    UNION ALL SELECT 'dropoff_centroid_latitude', 'is_null', dropoff_lat_null, total_rows, 'medium' FROM checks
    UNION ALL SELECT 'dropoff_centroid_latitude', 'outside broad Chicago range 41..43', dropoff_lat_outside_broad_chicago_range, total_rows, 'high' FROM checks
    UNION ALL SELECT 'dropoff_centroid_longitude', 'is_null', dropoff_lon_null, total_rows, 'medium' FROM checks
    UNION ALL SELECT 'dropoff_centroid_longitude', 'outside broad Chicago range -89..-87', dropoff_lon_outside_broad_chicago_range, total_rows, 'high' FROM checks
    UNION ALL SELECT 'dropoff_centroid_location', 'is_null', dropoff_location_null, total_rows, 'medium' FROM checks

    -- Derived Outlier Checks
    UNION ALL SELECT 'mph', '> 100 mph', mph_gt_100, total_rows, 'medium' FROM checks
    UNION ALL SELECT 'fare_per_mile', '> 50', fare_per_mile_gt_50, total_rows, 'medium' FROM checks
    UNION ALL SELECT 'total_per_mile', '> 100', total_per_mile_gt_100, total_rows, 'medium' FROM checks
)

SELECT
    column_name,
    check_name,
    n_rows,
    ROUND(100.0 * n_rows / total_rows, 4) AS pct,
    severity
FROM long
ORDER BY
    CASE severity
        WHEN 'high' THEN 1
        WHEN 'medium' THEN 2
        WHEN 'info' THEN 3
        ELSE 4
    END,
    n_rows DESC,
    column_name,
    check_name
""").df()

problem_checks = (
    plausi_report
    .query("n_rows > 0")
    .sort_values(["severity", "n_rows"], ascending=[True, False])
)

print(problem_checks.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

               column_name                                         check_name    n_rows     pct severity
                trip_miles                                               <= 0 1338256.0  8.6860     high
              trip_seconds                                               <= 0  208815.0  1.3553     high
                      fare                                               <= 0   39734.0  0.2579     high
                trip_total                                               <= 0   37776.0  0.2452     high
                      fare                                            is_null   31816.0  0.2065     high
                trip_total                                            is_null   31816.0  0.2065     high
              trip_seconds                                               > 6h    8822.0  0.0573     high
              trip_seconds                                            is_null    2936.0  0.0191     high
                      fare                             

In [ ]:
duckdb.sql(f"""
SELECT pickup_centroid_location, COUNT(*)
FROM read_parquet('{TAXI_PATH}')
GROUP BY pickup_centroid_location
ORDER BY count(*) desc
""").show()

┌──────────────────────────────────────┬──────────────┐
│       pickup_centroid_location       │ count_star() │
│               varchar                │    int64     │
├──────────────────────────────────────┼──────────────┤
│ POINT (-87.913624596 41.9802643146)  │      1559412 │
│ POINT (-87.9030396611 41.9790708201) │      1523891 │
│ POINT (-87.6333080367 41.899602111)  │      1483029 │
│ POINT (-87.6327464887 41.8809944707) │       737815 │
│ POINT (-87.6209929134 41.8849871918) │       721452 │
│ POINT (-87.6251921424 41.8788655841) │       704937 │
│ POINT (-87.6635175498 41.874005383)  │       622651 │
│ POINT (-87.642648998 41.8792550844)  │       560129 │
│ POINT (-87.6559981815 41.9442266014) │       478187 │
│ NULL                                 │       421354 │
│  ·                                   │            · │
│  ·                                   │            · │
│  ·                                   │            · │
│ POINT (-87.7142147077 41.8681324282) │        

## Uniqueness Check

In [ ]:
duckdb.sql(f"""
SELECT
    COUNT(*) AS n_rows,
    COUNT(DISTINCT trip_id) AS distinct_trip_id,
    COUNT(*) - COUNT(DISTINCT trip_id) AS duplicate_trip_id,

    COUNT(DISTINCT (taxi_id, trip_start_timestamp, trip_end_timestamp)) AS distinct_taxi_time_key,
    COUNT(*) - COUNT(DISTINCT (taxi_id, trip_start_timestamp, trip_end_timestamp)) AS duplicate_taxi_time_key
FROM read_parquet('{TAXI_PATH}')
""").show(max_width=500)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────┬──────────────────┬───────────────────┬────────────────────────┬─────────────────────────┐
│  n_rows  │ distinct_trip_id │ duplicate_trip_id │ distinct_taxi_time_key │ duplicate_taxi_time_key │
│  int64   │      int64       │       int64       │         int64          │          int64          │
├──────────┼──────────────────┼───────────────────┼────────────────────────┼─────────────────────────┤
│ 15406960 │         15406960 │                 0 │               15223605 │                  183355 │
└──────────┴──────────────────┴───────────────────┴────────────────────────┴─────────────────────────┘



## Coverage Check

In [ ]:
# Monthly
duckdb.sql(f"""
WITH base AS (
    SELECT TRY_CAST(trip_start_timestamp AS TIMESTAMP) AS trip_start
    FROM read_parquet('{TAXI_PATH}')
)
SELECT
    DATE_TRUNC('month', trip_start) AS month,
    COUNT(*) AS n_rows
FROM base
GROUP BY month
HAVING COUNT(*) < 500000
ORDER BY month
""").show(max_rows=100)

# Daily
duckdb.sql(f"""
WITH base AS (
    SELECT CAST(TRY_CAST(trip_start_timestamp AS TIMESTAMP) AS DATE) AS trip_date
    FROM read_parquet('{TAXI_PATH}')
)
SELECT
    trip_date,
    COUNT(*) AS n_rows
FROM base
GROUP BY trip_date
HAVING COUNT(*) < 10000
ORDER BY trip_date
""").show(max_rows=400)

# Missing days
coverage_report = duckdb.sql(f"""
WITH base AS (
    SELECT
        CAST(TRY_CAST(trip_start_timestamp AS TIMESTAMP) AS DATE) AS trip_date
    FROM read_parquet('{TAXI_PATH}')
    WHERE TRY_CAST(trip_start_timestamp AS TIMESTAMP) IS NOT NULL
),

date_bounds AS (
    SELECT
        MIN(trip_date) AS min_date,
        MAX(trip_date) AS max_date
    FROM base
),

date_spine AS (
    SELECT
        d::DATE AS trip_date
    FROM date_bounds,
    generate_series(min_date, max_date, INTERVAL 1 DAY) AS t(d)
),

daily_counts AS (
    SELECT
        trip_date,
        COUNT(*) AS n_rows
    FROM base
    GROUP BY trip_date
),

coverage AS (
    SELECT
        ds.trip_date,
        COALESCE(dc.n_rows, 0) AS n_rows,
        CASE
            WHEN dc.n_rows IS NULL THEN TRUE
            ELSE FALSE
        END AS is_missing_day
    FROM date_spine ds
    LEFT JOIN daily_counts dc
        ON ds.trip_date = dc.trip_date
)

SELECT
    trip_date,
    n_rows,
    is_missing_day
FROM coverage
ORDER BY trip_date
""").df()

missing_days = coverage_report.query("is_missing_day == True")

print(missing_days.to_string(index=False))

┌─────────────────────┬────────┐
│        month        │ n_rows │
│      timestamp      │ int64  │
├─────────────────────┼────────┤
│ 2024-01-01 00:00:00 │ 425203 │
│ 2024-02-01 00:00:00 │ 440013 │
│ 2025-01-01 00:00:00 │ 424927 │
│ 2025-02-01 00:00:00 │ 449874 │
│ 2026-01-01 00:00:00 │ 462991 │
│ 2026-02-01 00:00:00 │ 450152 │
│ 2026-05-01 00:00:00 │     34 │
└─────────────────────┴────────┘

┌────────────┬────────┐
│ trip_date  │ n_rows │
│    date    │ int64  │
├────────────┼────────┤
│ 2024-01-01 │   9511 │
│ 2024-01-06 │   9290 │
│ 2024-01-07 │   8786 │
│ 2024-01-13 │   9676 │
│ 2024-01-14 │   8536 │
│ 2024-01-27 │   9992 │
│ 2024-01-28 │   9290 │
│ 2024-02-04 │   9736 │
│ 2024-02-11 │   9712 │
│ 2024-07-04 │   9664 │
│ 2024-11-28 │   7860 │
│ 2024-12-24 │   9544 │
│ 2024-12-25 │   5326 │
│ 2025-01-01 │   9285 │
│ 2025-01-05 │   9154 │
│ 2025-01-11 │   9705 │
│ 2025-01-12 │   8865 │
│ 2025-01-18 │   9793 │
│ 2025-01-19 │   8560 │
│ 2025-01-26 │   9228 │
│ 2025-02-02 │   9800 │
│ 2

In [ ]:
low_volume_days = duckdb.sql(f"""
WITH base AS (
    SELECT
        CAST(TRY_CAST(trip_start_timestamp AS TIMESTAMP) AS DATE) AS trip_date
    FROM read_parquet('{TAXI_PATH}')
    WHERE TRY_CAST(trip_start_timestamp AS TIMESTAMP) IS NOT NULL
),

daily_counts AS (
    SELECT
        trip_date,
        COUNT(*) AS n_rows
    FROM base
    GROUP BY trip_date
),

stats AS (
    SELECT
        median(n_rows) AS median_daily_rows
    FROM daily_counts
)

SELECT
    dc.trip_date,
    dc.n_rows,
    s.median_daily_rows,
    ROUND(100.0 * dc.n_rows / s.median_daily_rows, 2) AS pct_of_median
FROM daily_counts dc
CROSS JOIN stats s
WHERE dc.n_rows < 0.5 * s.median_daily_rows
ORDER BY dc.trip_date
""").df()

print(low_volume_days.to_string(index=False))

 trip_date  n_rows  median_daily_rows  pct_of_median
2024-01-06    9290            18581.5          50.00
2024-01-07    8786            18581.5          47.28
2024-01-14    8536            18581.5          45.94
2024-01-28    9290            18581.5          50.00
2024-11-28    7860            18581.5          42.30
2024-12-25    5326            18581.5          28.66
2025-01-01    9285            18581.5          49.97
2025-01-05    9154            18581.5          49.26
2025-01-12    8865            18581.5          47.71
2025-01-19    8560            18581.5          46.07
2025-01-26    9228            18581.5          49.66
2025-11-27    9114            18581.5          49.05
2025-12-25    5220            18581.5          28.09
2026-05-01      34            18581.5           0.18


In [ ]:
duckdb.sql(f"""select min(trip_start_timestamp) from read_parquet('{TAXI_PATH}') """)

┌───────────────────────────┐
│ min(trip_start_timestamp) │
│         timestamp         │
├───────────────────────────┤
│ 2024-01-01 00:00:00       │
└───────────────────────────┘

## Weather Data

In [39]:
WEATHER_PATH = "bronze_weatherdata.parquet"

duckdb.sql(f"""
SELECT COUNT(*) AS n_rows
FROM read_parquet('{WEATHER_PATH}')
""").show()

duckdb.sql(f"""
DESCRIBE SELECT *
FROM read_parquet('{WEATHER_PATH}')
""").show(max_rows=100)

┌────────┐
│ n_rows │
│ int64  │
├────────┤
│  24360 │
└────────┘

┌─────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │ column_type │  null   │   key   │ default │  extra  │
│   varchar   │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ station     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ valid       │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ lon         │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ lat         │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ tmpc        │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ relh        │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ sknt        │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ p01m        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ vsby        │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ gust        │ DOUBLE 

### Column Description
station: weather station always MDW (metadata)
valid: timestamp
lon: longitude (metadata)
lat: latitude (metadata)

tmpc: temperature in celcius
relh: relative humitidy in %
sknt: wind speed in knots
p01m: precipitation in mm
vsby: visibility in miles
gust: wind gust in knots
skyc1: cloud coverage level 1
skyc2: cloud coverage level 2
skyc3: cloud coverage level 3



## Null Values

In [40]:
con = duckdb.connect()

columns = con.sql(f"""
DESCRIBE SELECT *
FROM read_parquet('{WEATHER_PATH}')
""").df()["column_name"].tolist()

parts = []

for col in columns:
    null_condition = f"""
        "{col}" IS NULL
        OR LOWER(TRIM(CAST("{col}" AS VARCHAR))) = 'null'
    """

    parts.append(f"""
    SELECT
        '{col}' AS column_name,
        COUNT(*) AS n_rows,
        SUM(CASE WHEN {null_condition} THEN 1 ELSE 0 END) AS n_nulls,
        ROUND(
            100.0 * SUM(CASE WHEN {null_condition} THEN 1 ELSE 0 END) / COUNT(*),
            2
        ) AS null_pct
    FROM read_parquet('{WEATHER_PATH}')
    """)

query = "\nUNION ALL\n".join(parts) + "\nORDER BY null_pct DESC"

null_report = con.sql(query).df()
print(null_report.to_string(index=False))

column_name  n_rows  n_nulls  null_pct
       gust   24360  18401.0     75.54
      skyc3   24360  18227.0     74.82
      skyc2   24360  11411.0     46.84
       p01m   24360      3.0      0.01
       relh   24360      2.0      0.01
       tmpc   24360      2.0      0.01
    station   24360      0.0      0.00
      skyc1   24360      0.0      0.00
        lon   24360      0.0      0.00
      valid   24360      0.0      0.00
       vsby   24360      0.0      0.00
       sknt   24360      1.0      0.00
        lat   24360      0.0      0.00


## Coverage Check

- Min and max timestamp
- are there multiple timestamp for each hour?
- is there a timestamp for each hour between min and max timestamp?

In [ ]:

duckdb.sql(f"""SELECT MIN(valid) from read_parquet('{WEATHER_PATH}')""").show()
duckdb.sql(f"""SELECT MAX(valid) from read_parquet('{WEATHER_PATH}')""").show()

┌─────────────────────┐
│    min("valid")     │
│      timestamp      │
├─────────────────────┤
│ 2024-01-01 00:53:00 │
└─────────────────────┘

┌─────────────────────┐
│    max("valid")     │
│      timestamp      │
├─────────────────────┤
│ 2026-05-25 23:53:00 │
└─────────────────────┘



In [47]:
duckdb.sql(f"""
    SELECT date_trunc('hour', valid) AS hour_ts, count(*)
    FROM read_parquet('{WEATHER_PATH}')
    GROUP BY date_trunc('hour', valid)
    HAVING count(*) > 1
    ORDER BY count(*) desc
""").show()

┌─────────────────────┬──────────────┐
│       hour_ts       │ count_star() │
│      timestamp      │    int64     │
├─────────────────────┼──────────────┤
│ 2026-01-31 00:00:00 │            7 │
│ 2025-03-05 19:00:00 │            6 │
│ 2024-01-25 13:00:00 │            6 │
│ 2026-04-04 10:00:00 │            6 │
│ 2026-03-11 18:00:00 │            6 │
│ 2026-05-18 16:00:00 │            6 │
│ 2024-01-24 01:00:00 │            6 │
│ 2026-01-14 13:00:00 │            6 │
│ 2025-12-31 22:00:00 │            6 │
│ 2026-04-03 01:00:00 │            6 │
│          ·          │            · │
│          ·          │            · │
│          ·          │            · │
│ 2026-04-18 02:00:00 │            2 │
│ 2024-01-10 07:00:00 │            2 │
│ 2026-04-26 10:00:00 │            2 │
│ 2026-04-28 03:00:00 │            2 │
│ 2026-04-28 04:00:00 │            2 │
│ 2026-05-01 14:00:00 │            2 │
│ 2026-05-04 21:00:00 │            2 │
│ 2025-02-13 12:00:00 │            2 │
│ 2026-05-23 11:00:00 │  

In [41]:
duckdb.sql(f"""-- Pfad anpassen
WITH raw AS (
    SELECT
        CAST(valid AS TIMESTAMP) AS valid_ts
    FROM read_parquet('{WEATHER_PATH}')
),

bounds AS (
    SELECT
        date_trunc('hour', min(valid_ts)) AS start_hour,
        date_trunc('hour', max(valid_ts)) AS end_hour
    FROM raw
),

expected_hours AS (
    SELECT generate_series AS hour_ts
    FROM bounds,
    generate_series(start_hour, end_hour, INTERVAL '1 hour')
),

observed_hours AS (
    SELECT
        date_trunc('hour', valid_ts) AS hour_ts,
        count(*) AS observations_in_hour
    FROM raw
    GROUP BY 1
),

hourly_check AS (
    SELECT
        e.hour_ts,
        COALESCE(o.observations_in_hour, 0) AS observations_in_hour,
        CASE
            WHEN o.hour_ts IS NULL THEN false
            ELSE true
        END AS has_data
    FROM expected_hours e
    LEFT JOIN observed_hours o
        ON e.hour_ts = o.hour_ts
)

SELECT *
FROM hourly_check
WHERE has_data = false
ORDER BY hour_ts;""").show()

┌─────────────────────┬──────────────────────┬──────────┐
│       hour_ts       │ observations_in_hour │ has_data │
│      timestamp      │        int64         │ boolean  │
├─────────────────────┼──────────────────────┼──────────┤
│ 2025-03-10 04:00:00 │                    0 │ false    │
│ 2025-09-24 02:00:00 │                    0 │ false    │
│ 2025-09-24 03:00:00 │                    0 │ false    │
│ 2025-09-24 04:00:00 │                    0 │ false    │
│ 2025-09-29 13:00:00 │                    0 │ false    │
│ 2026-01-16 12:00:00 │                    0 │ false    │
│ 2026-04-03 06:00:00 │                    0 │ false    │
│ 2026-04-03 07:00:00 │                    0 │ false    │
│ 2026-04-03 09:00:00 │                    0 │ false    │
│ 2026-04-03 10:00:00 │                    0 │ false    │
│ 2026-05-18 04:00:00 │                    0 │ false    │
│ 2026-05-18 05:00:00 │                    0 │ false    │
└─────────────────────┴──────────────────────┴──────────┘
  12 rows     